In [0]:
%pip install vaderSentiment -q

In [0]:
import logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
from delta.tables import DeltaTable


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

BRONZE_PATH = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/JSON_Streaming_Data_Source_2//datafiles/"
SILVER_PATH = "abfss://silverlayer@cryptodl.dfs.core.windows.net/JSON_Streaming_Data_Source_2/"

spark = SparkSession.builder.getOrCreate()


log.info("Reading Bronze Source 2...")
df_bronze = spark.read.format("delta").load(BRONZE_PATH)
log.info(f"Bronze rows loaded: {df_bronze.count():,}")
display(df_bronze.limit(5))

In [0]:
log.info("Cleaning data...")
df_clean = (
    df_bronze
    .drop("_rescued_data")
    .withColumn("trade_date",      F.to_date(F.col("trade_date")))
    .withColumn("event_timestamp", F.to_timestamp(F.col("event_timestamp")))
    .filter(F.col("text").isNotNull() & (F.trim(F.col("text")) != ""))
    .dropDuplicates(["post_id"])
)
log.info(f"After cleaning: {df_clean.count():,}")
display(df_clean.limit(5))

In [0]:
# NLP Sentiment Analysis
log.info("Running VADER sentiment analysis...")

@F.udf(returnType=DoubleType())
def sentiment_score_udf(text):
    if not text:
        return 0.0
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    return float(SentimentIntensityAnalyzer().polarity_scores(text)["compound"])

@F.udf(returnType=StringType())
def sentiment_label_udf(score):
    if score is None:  return "neutral"
    if score >= 0.05:  return "positive"
    if score <= -0.05: return "negative"
    return "neutral"

df_nlp = (
    df_clean
    .withColumn("sentiment_score",     sentiment_score_udf(F.col("text")))
    .withColumn("sentiment_label",     sentiment_label_udf(F.col("sentiment_score")))
    .withColumn("silver_processed_at", F.current_timestamp())
)

df_final = df_nlp.select(
    "post_id", "symbol", "trade_date", "source_type",
    "text", "sentiment_score", "sentiment_label",
    "engagement_score", "event_timestamp", "silver_processed_at"
)

log.info(f"NLP done: {df_final.count():,} rows")
display(df_final.select("symbol","text","sentiment_score","sentiment_label").limit(10))

In [0]:
# Write Silver Delta (Create or Merge)

log.info(f"Writing Silver Delta to: {SILVER_PATH}")

if DeltaTable.isDeltaTable(spark, SILVER_PATH):
    log.info("Delta table exists — running MERGE...")
    (
        DeltaTable.forPath(spark, SILVER_PATH)
        .alias("target")
        .merge(df_final.alias("source"), "target.post_id = source.post_id")
        .whenMatchedUpdate(set={
            "sentiment_score":     "source.sentiment_score",
            "sentiment_label":     "source.sentiment_label",
            "silver_processed_at": "source.silver_processed_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    log.info("MERGE complete")

else:
    log.info("Delta table not found — creating (first run)...")
    (
        df_final.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("symbol", "trade_date")
        .save(SILVER_PATH)
    )
    log.info("Delta table created")


In [0]:
log.info("Verifying Silver Delta...")
df_verify = spark.read.format("delta").load(SILVER_PATH)

log.info(f"Total rows    : {df_verify.count():,}")
log.info(f"Unique symbols: {df_verify.select('symbol').distinct().count()}")

df_verify.groupBy("sentiment_label").count().orderBy("sentiment_label").show()

display(
    df_verify.select("symbol","trade_date","text","sentiment_score","sentiment_label","engagement_score")
             .limit(20)
)